In [ ]:
!pip -q install transformers datasets accelerate scikit-learn langdetect vaderSentiment

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import re, numpy as np, pandas as pd
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

URL = "hf://datasets/AmaanP314/youtube-comment-sentiment/youtube-comments-sentiment.csv"
df = pd.read_csv(URL, usecols=["CommentID", "VideoID", "CommentText", "Sentiment"])
df = df[~df["CommentID"].astype(str).str.endswith("_aug")]
df = df.dropna(subset=["CommentText", "Sentiment"])
df["text"] = df["CommentText"].astype(str).str.strip()
df["label"] = df["Sentiment"].str.lower()
df = df[df["label"].isin(["negative", "neutral", "positive"])]
df = df[df["text"].str.len() > 1].drop_duplicates(subset="text")

def mostly_latin(t, max_foreign=0.3):
    letters = [c for c in t if c.isalpha()]
    if not letters:
        return True
    return sum(ord(c) > 0x024F for c in letters) / len(letters) <= max_foreign
df = df[df["text"].map(mostly_latin)]

# test comments come from videos that are never used for training
rng = np.random.RandomState(42)
vids = df["VideoID"].unique()
test_vids = set(rng.choice(vids, size=int(0.05 * len(vids)), replace=False))
test_df = df[df["VideoID"].isin(test_vids)]
train_df = df[~df["VideoID"].isin(test_vids)]

TRAIN_PER_CLASS = 20000
TEST_SIZE = 6000
train_df = pd.concat([g.sample(min(len(g), TRAIN_PER_CLASS), random_state=1)
                      for _, g in train_df.groupby("label")])
test_df = test_df.sample(min(len(test_df), TEST_SIZE), random_state=1)

def is_en(t):
    try:
        return detect(t) == "en"
    except Exception:
        return False
train_df = train_df[train_df["text"].map(is_en)]
test_df = test_df[test_df["text"].map(is_en)].copy()

print(len(train_df), "train |", len(test_df), "test")
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

51699 train | 5186 test
label
negative    17806
positive    17562
neutral     16331
Name: count, dtype: int64
label
positive    2103
negative    1580
neutral     1503
Name: count, dtype: int64


In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

BASE = "cardiffnlp/twitter-roberta-base-sentiment-latest"
LABELS = ["negative", "neutral", "positive"]
l2i = {l: i for i, l in enumerate(LABELS)}

def prep(t):
    t = re.sub(r"https?://\S+", "http", str(t))
    t = re.sub(r"@[\w.\-]+", "@user", t)
    return re.sub(r"\s+", " ", t).strip()

tr, va = train_test_split(train_df, test_size=0.05, stratify=train_df["label"], random_state=42)
tok = AutoTokenizer.from_pretrained(BASE)

def to_ds(d):
    d = pd.DataFrame({"text": d["text"].map(prep), "label": d["label"].map(l2i)})
    ds = Dataset.from_pandas(d.reset_index(drop=True))
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=96), batched=True)

train_ds, val_ds, test_ds = to_ds(tr), to_ds(va), to_ds(test_df)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=3,
    id2label={i: l for l, i in l2i.items()}, label2id=l2i)

def metrics(p):
    pred = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(p.label_ids, pred),
            "precision_macro": pr, "recall_macro": rc, "f1_macro": f1}

args = TrainingArguments(
    output_dir="yt-sentiment", num_train_epochs=2,
    learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=128,
    weight_decay=0.01, warmup_steps=300, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    save_total_limit=1, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)

def report(name, y_true, y_pred):
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n=== {name} ===")
    print("accuracy:", round(accuracy_score(y_true, y_pred), 4),
          "| macro precision:", round(pr, 4), "| macro recall:", round(rc, 4), "| macro F1:", round(f1, 4))
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=4, zero_division=0))
    print(confusion_matrix(y_true, y_pred))

y_true = test_df["label"].map(l2i).values

before = trainer.predict(test_ds)           # original model, before any training
report("ORIGINAL MODEL (no training)", y_true, before.predictions.argmax(-1))

trainer.train()

after = trainer.predict(test_ds)
after_pred = after.predictions.argmax(-1)
report("YOUR FINE-TUNED MODEL", y_true, after_pred)

sia = SentimentIntensityAnalyzer()
def vader(t):
    c = sia.polarity_scores(t)["compound"]
    return l2i["positive"] if c >= 0.05 else l2i["negative"] if c <= -0.05 else l2i["neutral"]
report("VADER BASELINE", y_true, [vader(t) for t in test_df["text"]])

test_df["pred"] = [LABELS[i] for i in after_pred]
trainer.save_model("yt-sentiment-final")
tok.save_pretrained("yt-sentiment-final")

Map:   0%|          | 0/49114 [00:00<?, ? examples/s]

Map:   0%|          | 0/2585 [00:00<?, ? examples/s]

Map:   0%|          | 0/5186 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== YOUR FINE-TUNED MODEL ===
accuracy: 0.7601 | macro precision: 0.7523 | macro recall: 0.7531 | macro F1: 0.7511
              precision    recall  f1-score   support

    negative     0.7122    0.8114    0.7586      1580
     neutral     0.7018    0.6420    0.6706      1503
    positive     0.8429    0.8060    0.8240      2103

    accuracy                         0.7601      5186
   macro avg     0.7523    0.7531    0.7511      5186
weighted avg     0.7622    0.7601    0.7596      5186

[[1282  209   89]
 [ 311  965  227]
 [ 207  201 1695]]

=== VADER BASELINE ===
accuracy: 0.5605 | macro precision: 0.5503 | macro recall: 0.5346 | macro F1: 0.5329
              precision    recall  f1-score   support

    negative     0.5810    0.4835    0.5278      1580
     neutral     0.4926    0.3553    0.4128      1503
    positive     0.5773    0.7651    0.6581      2103

    accuracy                         0.5605      5186
   macro avg     0.5503    0.5346    0.5329      5186
weighted avg 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('yt-sentiment-final/tokenizer_config.json',
 'yt-sentiment-final/tokenizer.json')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
from transformers import AutoModelForSequenceClassification, AutoTokenizer
m = AutoModelForSequenceClassification.from_pretrained("yt-sentiment-final")
t = AutoTokenizer.from_pretrained("yt-sentiment-final")
m.push_to_hub("roshan050/yt-sentiment-roberta")
t.push_to_hub("roshan050/yt-sentiment-roberta")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ygzrxxt/model.safetensors:   0%|          |  551kB /  499MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/roshan050/yt-sentiment-roberta/commit/c9c3e69728e42dc6167e9040cdfe58231282b27b', commit_message='Upload tokenizer', commit_description='', oid='c9c3e69728e42dc6167e9040cdfe58231282b27b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/roshan050/yt-sentiment-roberta', endpoint='https://huggingface.co', repo_type='model', repo_id='roshan050/yt-sentiment-roberta'), pr_revision=None, pr_num=None)

In [ ]:
!pip -q install gradio_client
from gradio_client import Client
c = Client("roshan050/sentiment-analysis-gradio")
print(c.predict(["great video, loved it!", "this is trash, waste of time", "what camera do you use?"],
                api_name="/predict"))

Loaded as API: https://roshan050-sentiment-analysis-gradio.hf.space
[{'label': 'positive', 'scores': {'negative': 0.0005, 'neutral': 0.0013, 'positive': 0.9982}}, {'label': 'negative', 'scores': {'negative': 0.9902, 'neutral': 0.0028, 'positive': 0.007}}, {'label': 'neutral', 'scores': {'negative': 0.0022, 'neutral': 0.9932, 'positive': 0.0046}}]
